In [ ]:
# P3-KN04: method 1 v1.2 (A9 cap on the whole prefix), 7 prefix settings x 2 parsers, then bge-m3 vectors for new texts only
!git clone -q https://github.com/amanchaurasia512/Self_optimized_Agent_Ai_chunking.git /kaggle/working/repo
!cd /kaggle/working/repo && git log --oneline -1
!pip install -q docling==2.143.0 jsonschema langchain-text-splitters==1.1.3 langchain-experimental==0.4.2 langchain-community==0.4.2 sentence-transformers==6.2.0 transformers==5.22.1 stanza==1.15.0 google-genai==2.34.0
import torch
print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
import os
import shutil
import sys
import zipfile
from pathlib import Path

from kaggle_secrets import UserSecretsClient

REPO, INPUT = Path("/kaggle/working/repo"), Path("/kaggle/input")
sys.path.insert(0, str(REPO))
from src.common.split import tuning_ids

tuning = tuning_ids()
for archive in INPUT.rglob("*.zip"):
    zipfile.ZipFile(archive).extractall(Path("/kaggle/temp") / archive.stem)
roots = [INPUT, Path("/kaggle/temp")]
for parser in ["docling", "pdfplumber"]:
    target = REPO / "data" / "extracted" / parser
    target.mkdir(parents=True, exist_ok=True)
    sources = [path for root in roots for path in root.rglob(f"{parser}/*.json") if path.stem in tuning]
    for path in sources:
        shutil.copyfile(path, target / path.name)
    print(parser, ":", len({path.stem for path in sources}), "of", len(tuning), "tuning extractions copied")

GOLD = next(path for root in roots for path in root.rglob("gold") if list(path.glob("spans_*.jsonl")))
CACHE = Path("/kaggle/working/embeddings/bge-m3")
source = next(Path(folder) for root in roots for folder, subfolders, files in os.walk(root, followlinks=True)
              if "embedder.json" in files and "embeddings_test" not in folder)
shutil.rmtree(CACHE, ignore_errors=True)
shutil.copytree(source, CACHE)
print("gold:", GOLD, "| cache copied from", source, "|", len(list(CACHE.glob("*.npy"))), "vector part(s)")

key_file = Path.home() / ".config" / "gemini" / "api_key"
key_file.parent.mkdir(parents=True, exist_ok=True)
key_file.write_text(UserSecretsClient().get_secret("GEMINI_API_KEY").strip())
key_file.chmod(0o600)

In [ ]:
import subprocess

RUNS = [("method1_tree", f"T={size} --heading-prefix") for size in [256, 384, 512, 640, 768, 896, 1024]]
print(len(RUNS), "settings")
failed = []
for chunker, settings in RUNS:
    command = f"python -m scripts.p2_chunk --chunker {chunker} --set {settings} --parsers docling pdfplumber --machine kaggle --run-source P3-KN04"
    result = subprocess.run(command, shell=True, cwd=REPO, capture_output=True, text=True)
    lines = (result.stdout + result.stderr).strip().splitlines()
    print(f"=== {chunker} {settings} | exit {result.returncode}\n" + "\n".join(lines[:1] + lines[-3:]) + "\n")
    if result.returncode:
        failed.append(f"{chunker} {settings}")
print("failed:", failed or "none")

In [ ]:
!cd /kaggle/working/repo && python -m scripts.p3_embed --chunks /kaggle/working/repo/data/chunks --gold {GOLD} --cache {CACHE} --name P3-KN04-prefix --run-source KN04